# Cellware cloud runner

Runs a sweep file from `sweeps/` on Colab (or any Linux box with Python). The networks are small, so CPU cores matter more than a GPU: a CPU runtime is fine; a high-RAM or Pro runtime gives more cores.

1. Set `SWEEP` below.
2. Runtime > Run all.
3. Results are copied to Google Drive (`MyDrive/cellware_runs/<sweep>/`) and offered as a zip download. Sweeps resume: rerunning skips finished runs if the Drive copy is restored first (cell 3).

In [ ]:
SWEEP = 'sweeps/e5_20seeds.txt'   # any file in sweeps/
BRANCH = 'modernize'
USE_DRIVE = True
NEED_LARRY_DATA = False      # True for fit_larry.py / larry_*.py sweeps

In [ ]:
import os, subprocess
if not os.path.exists('cellware'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', 'https://github.com/krenusz/cellware.git'], check=True)
os.chdir('cellware')
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.[fit]'], check=True)  # torch is preinstalled on Colab
print('cores:', os.cpu_count())

In [ ]:
import shutil, glob
name = os.path.splitext(os.path.basename(SWEEP))[0]
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    store = f'/content/drive/MyDrive/cellware_runs/{name}'
    os.makedirs(store, exist_ok=True)
    # restore finished runs so the sweep resumes instead of repeating them
    for f in glob.glob(f'{store}/**/*.json', recursive=True):
        dst = os.path.relpath(f, store)
        os.makedirs(os.path.dirname(dst) or '.', exist_ok=True)
        shutil.copy(f, dst)

In [ ]:
if NEED_LARRY_DATA:
    base = 'https://kleintools.hms.harvard.edu/paper_websites/state_fate2020/'
    for sub, prefix in [('larry', 'stateFate_inVitro_'), ('larry_invivo', 'stateFate_inVivo_')]:
        os.makedirs(f'data/{sub}', exist_ok=True)
        for f in ['clone_matrix.mtx.gz', 'metadata.txt.gz']:
            subprocess.run(['wget', '-q', '-nc', '-O', f'data/{sub}/{f}', base + prefix + f])

In [ ]:
workers = max(1, os.cpu_count() // 2)   # 2 threads per job
!python experiments/sweep.py {SWEEP} --workers {workers} --threads 2

In [ ]:
outs = sorted({os.path.dirname(l.split(']')[0][1:]) for l in open(SWEEP) if l.startswith('[')})
print('result folders:', outs)
export = f'/content/export_{name}'
for d in outs + [f'experiments/logs_sweep/{name}']:
    shutil.copytree(d, f'{export}/{d}', dirs_exist_ok=True)
if USE_DRIVE:
    shutil.copytree(export, store, dirs_exist_ok=True)
shutil.make_archive(f'/content/{name}', 'zip', export)
from google.colab import files
files.download(f'/content/{name}.zip')  # unzip into the repo root locally